In [2]:
import pandas as pd

df_raw = pd.read_csv("listings (1).csv.gz")
print("Gesamtanzahl Listings (roh):", len(df_raw))

Gesamtanzahl Listings (roh): 14274


In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# ==============================================================================
# 0. DATEN LADEN & PREIS KONVERTIEREN (Das hat gefehlt!)
# ==============================================================================
# Die vollständige Liste aller Spalten, die wir für die Analysen brauchen
relevant_columns = [
    'id', 'listing_url', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'room_type', 'property_type', 'accommodates',
    'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 
    'review_scores_rating', 'calculated_host_listings_count', 'availability_365',
    'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location', 'review_scores_value',
    'host_is_superhost',  'host_response_rate', 'host_acceptance_rate', 'bathrooms_text', 'host_response_time',
    'instant_bookable', 'amenities', 'beds', 'bedrooms', 'host_since', 'availability_30', 'description', 'host_identity_verified',
    'neighbourhood', 'bathrooms', 'bathrooms_text', 'host_response_time', 'number_of_reviews_ltm'
]
print("Lade Daten aus der CSV...")
# Lädt die Datei (achte darauf, dass der Dateiname exakt stimmt, z.B. 'listings (1).csv.gz' oder 'listings.csv')
df_raw = pd.read_csv('listings (1).csv.gz', usecols=relevant_columns)

df = df_raw.copy()  # Wir arbeiten mit einer Kopie, um die Originaldaten zu behalten

# Preis von Text (z.B. "$120.00") in eine echte Zahl umwandeln
df['price'] = df_raw['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')


# ==============================================================================
# 1. BASIS-RELEVANZFILTER (Für beide Methoden identisch)
# ==============================================================================
# Wir behalten nur aktive Listings mit Bewertungen, um Karteileichen auszuschließen
df_active = df[
    (df['price'] > 0) & 
    (df['number_of_reviews'] >= 3) & 
    (df['review_scores_rating'].notna())
].copy()

print(f"Basis-Datensatz (aktiv & bewertet): {df_active.shape[0]} Listings.\n")


# ==============================================================================
# METHODE A: DER HARTE CUT (95. Perzentil über den gesamten Datensatz)
# ==============================================================================
price_95_limit = df_active['price'].quantile(0.95) # Globales 95. Perzentil
df_95_cut = df_active[df_active['price'] <= price_95_limit].copy()

print(f"--- METHODE A (Harter Cut) ---")
print(f"Globales 95. Perzentil liegt bei: {price_95_limit:.2f} €")
print(f"Verbleibende Listings: {df_95_cut.shape[0]}")


# ==============================================================================
# METHODE B: DER IQR-CUT (Kontextsensitiv pro Zimmertyp)
# ==============================================================================
df_iqr_cut = pd.DataFrame()
print(f"\n--- METHODE B (IQR-Cut pro Zimmertyp) ---")

for r_type in df_active['room_type'].unique():
    subset = df_active[df_active['room_type'] == r_type]
    
    Q1 = subset['price'].quantile(0.25)
    Q3 = subset['price'].quantile(0.75)
    IQR = Q3 - Q1
    
    # 1. Reine mathematische Grenzen berechnen
    pure_upper_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR
    
    # 2. Heuristischen Schutzdeckel anwenden
    final_upper_bound = pure_upper_bound
    if final_upper_bound < 150: 
        final_upper_bound = 150
        
    print(f" - {r_type}:")
    print(f"   -> Info-Untergrenze (reine Formel): {lower_bound:.2f} €")
    print(f"   -> Reine Obergrenze (reine Formel): {pure_upper_bound:.2f} €")
    print(f"   -> FINALE FILTER-GRENZE (mit Deckel): {final_upper_bound:.2f} €")
    print("-" * 50)
    cleaned_subset = subset[subset['price'] <= final_upper_bound]
    df_iqr_cut = pd.concat([df_iqr_cut, cleaned_subset])

print(f"Verbleibende Listings gesamt: {df_iqr_cut.shape[0]}")

# 0/1 Dummies für Superhost und Instant Bookable direkt für die folgenden Regressionen vorbereiten
df_iqr_cut['is_superhost'] = df_iqr_cut['host_is_superhost'].map({'t': 1, 'f': 0})
df_iqr_cut['is_instant_bookable'] = df_iqr_cut['instant_bookable'].map({'t': 1, 'f': 0})

df_95_cut['is_superhost'] = df_95_cut['host_is_superhost'].map({'t': 1, 'f': 0})
df_95_cut['is_instant_bookable'] = df_95_cut['instant_bookable'].map({'t': 1, 'f': 0})

Lade Daten aus der CSV...
Basis-Datensatz (aktiv & bewertet): 6377 Listings.

--- METHODE A (Harter Cut) ---
Globales 95. Perzentil liegt bei: 301.00 €
Verbleibende Listings: 6060

--- METHODE B (IQR-Cut pro Zimmertyp) ---
 - Entire home/apt:
   -> Info-Untergrenze (reine Formel): -49.00 €
   -> Reine Obergrenze (reine Formel): 327.00 €
   -> FINALE FILTER-GRENZE (mit Deckel): 327.00 €
--------------------------------------------------
 - Private room:
   -> Info-Untergrenze (reine Formel): 1.50 €
   -> Reine Obergrenze (reine Formel): 141.50 €
   -> FINALE FILTER-GRENZE (mit Deckel): 150.00 €
--------------------------------------------------
 - Shared room:
   -> Info-Untergrenze (reine Formel): -2.38 €
   -> Reine Obergrenze (reine Formel): 86.62 €
   -> FINALE FILTER-GRENZE (mit Deckel): 150.00 €
--------------------------------------------------
 - Hotel room:
   -> Info-Untergrenze (reine Formel): -5.00 €
   -> Reine Obergrenze (reine Formel): 387.00 €
   -> FINALE FILTER-GRENZE 

In [6]:
# ==============================================================================
# 1. ADJUSTED IQR-CUT (Hubert & Vandervieren für rechtsschiefe Daten)
# ==============================================================================
print("--- ADJUSTED IQR-BEREINIGUNG (Medcouple pro Zimmertyp) ---")


def calculate_medcouple(y):
  """Berechnet das Medcouple (robustes Schiefemaß)."""
  if len(y) == 0:
    return 0.0
  y = np.sort(np.asarray(y, dtype=float))
  med = np.median(y)
  scale = 2 * np.max(np.abs(y))
  if scale == 0:
    return 0.0

  x_i = y[y <= med][:, None]
  x_j = y[y >= med][None, :]
  denom = x_j - x_i
  denom[denom == 0] = np.inf
  num = (x_j - med) - (med - x_i)
  h_matrix = num / denom
  return float(np.median(h_matrix))


df_iqr_cut_adj = pd.DataFrame()

for r_type in df_active["room_type"].unique():
  subset = df_active[df_active["room_type"] == r_type].copy()
  prices = subset["price"].values

  Q1 = subset["price"].quantile(0.25)
  Q3 = subset["price"].quantile(0.75)
  IQR = Q3 - Q1

  mc = calculate_medcouple(prices)

  # Whisker-Korrektur nach Hubert & Vandervieren (2008)
  # Bei Rechtsschiefe (mc >= 0): Obergrenze wird gestreckt, Untergrenze gestaucht
  if mc >= 0:
    upper_factor = np.exp(4.0 * mc)
    lower_factor = np.exp(-4.0 * mc)
    adj_upper_bound = Q3 + 1.5 * upper_factor * IQR
    adj_lower_bound = Q1 - 1.5 * lower_factor * IQR
  else:
    upper_factor = np.exp(3.0 * mc)
    lower_factor = np.exp(-3.0 * mc)
    adj_upper_bound = Q3 + 1.5 * upper_factor * IQR
    adj_lower_bound = Q1 - 1.5 * lower_factor * IQR

  print(f" - {r_type}:")
  print(f"   -> Q1: {Q1:.2f} € | Q3: {Q3:.2f} € | IQR: {IQR:.2f} €")
  print(
      f"   -> Medcouple (mc): {mc:.3f} | Oberer Faktor: {upper_factor:.2f}x |"
      f" Unterer Faktor: {lower_factor:.2f}x"
  )
  print(f"   -> Rechnerische UNTERGRENZE: {adj_lower_bound:.2f} €")
  print(f"   -> Berechnete OBERGRENZE:     {adj_upper_bound:.2f} €")

  # Da unplausible 0 €-Preise bereits in Stufe 1 entfernt wurden,
  # greift hier nur der Cutoff nach oben:
  cleaned_subset = subset[subset["price"] <= adj_upper_bound]
  df_iqr_cut_adj = pd.concat([df_iqr_cut_adj, cleaned_subset])

print("-" * 50)
print(f"Ausgangsdatensatz: {df_active.shape[0]} Zeilen")
print(
    f"Entfernte Ausreißer: {df_active.shape[0] - df_iqr_cut_adj.shape[0]}"
    " Inserate"
)
print(f"Bereinigter Datensatz: {df_iqr_cut_adj.shape[0]} Zeilen\n")

# ==============================================================================
# BINAERE FLAGS DEFINIEREN & EXPORTIEREN
# ==============================================================================
# Mapping der Plattform-Flags ('t'/'f' bzw. Boolean zu 1/0)
if "host_is_superhost" in df_iqr_cut_adj.columns:
  df_iqr_cut_adj["is_superhost"] = (
      df_iqr_cut_adj["host_is_superhost"]
      .map({"t": 1, "f": 0, True: 1, False: 0})
      .fillna(0)
      .astype(int)
  )

if "instant_bookable" in df_iqr_cut_adj.columns:
  df_iqr_cut_adj["is_instant_bookable"] = (
      df_iqr_cut_adj["instant_bookable"]
      .map({"t": 1, "f": 0, True: 1, False: 0})
      .fillna(0)
      .astype(int)
  )

# Datensatz für nachfolgende Skripte abspeichern
df_iqr_cut_adj.to_pickle("df_iqr_cut_adj.pkl")
print("Gespeichert als: df_iqr_cut_adj.pkl")

--- ADJUSTED IQR-BEREINIGUNG (Medcouple pro Zimmertyp) ---
 - Entire home/apt:
   -> Q1: 92.00 € | Q3: 186.00 € | IQR: 94.00 €
   -> Medcouple (mc): 0.267 | Oberer Faktor: 2.91x | Unterer Faktor: 0.34x
   -> Rechnerische UNTERGRENZE: 43.60 €
   -> Berechnete OBERGRENZE:     596.78 €
 - Private room:
   -> Q1: 54.00 € | Q3: 89.00 € | IQR: 35.00 €
   -> Medcouple (mc): 0.238 | Oberer Faktor: 2.59x | Unterer Faktor: 0.39x
   -> Rechnerische UNTERGRENZE: 33.74 €
   -> Berechnete OBERGRENZE:     225.07 €
 - Shared room:
   -> Q1: 31.00 € | Q3: 53.25 € | IQR: 22.25 €
   -> Medcouple (mc): 0.462 | Oberer Faktor: 6.34x | Unterer Faktor: 0.16x
   -> Rechnerische UNTERGRENZE: 25.73 €
   -> Berechnete OBERGRENZE:     264.69 €
 - Hotel room:
   -> Q1: 142.00 € | Q3: 240.00 € | IQR: 98.00 €
   -> Medcouple (mc): 0.141 | Oberer Faktor: 1.76x | Unterer Faktor: 0.57x
   -> Rechnerische UNTERGRENZE: 58.51 €
   -> Berechnete OBERGRENZE:     498.82 €
--------------------------------------------------
Aus